# Comparing ways to discretize a trajectory: the alanine pentapeptide

MasterMSM offers several routes from a molecular dynamics trajectory to a discrete state trajectory:

* **Buchete-Hummer (BH)**: assign each residue to a Ramachandran region (A, E, ...). Simple and interpretable, but specific to peptides.
* **Features + PCA + $k$-means**: describe each frame with features (here torsions), project on the directions of largest *variance* and cluster.
* **Features + TICA + $k$-means**: as above, but project on the directions of slowest *decorrelation*, which is what an MSM tries to resolve.
* **Features + TICA/PCA + HDBSCAN**: density-based clustering, which finds metastable core regions directly (optional, requires the `hdbscan` package).

Rather than running each by hand, overwriting `features` and `distraj` as we go, we describe each route as a *protocol* (a small dictionary) and let `compare_protocols` run all of them on the same data, without modifying the original trajectory. We then build an MSM from each one and compare the results on equal terms.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="ticks", color_codes=True, font_scale=1.2)
sns.set_style({"xtick.direction": "in", "ytick.direction": "in"})

We download the trajectory from the [OSF repository](https://osf.io/a2vc7/) if it is not already present, and load it.

In [ ]:
from mastermsm.test.download_data import download_osf_ala5, _DATA_DIR
from mastermsm.trajectory import traj
download_osf_ala5()
tr = traj.TimeSeries(top=os.path.join(_DATA_DIR, 'ala5.gro'),
                     xtc=os.path.join(_DATA_DIR, 'ala5.xtc'))
print(tr.mdt, "\ndt =", tr.dt, "ps")

### Describing the protocols
Each protocol is a dictionary with up to four steps: `features`, `reduce`, `cluster`, and `whiten` (on by default). Buchete-Hummer is selected with the `rama` key instead. The TICA lag is given in frames: 200 frames corresponds to 2 ns here.

We use the same torsion features ($\sin$ and $\cos$ of $\phi$ and $\psi$) and the same dimensionality (4) for PCA and TICA, so that the only difference between them is the criterion used to choose the directions.

In [ ]:
torsions = [('torsions', {'sincos': True})]
protocols = {
    'BH (A/E)':  {'rama': ['A', 'E']},
    'BH (A/E/L)': {'rama': ['A', 'E', 'L']},
    'PCA + kmeans':  {'features': torsions,
                      'reduce': ('pca', {'n': 4}),
                      'cluster': ('kmeans', {'k': 100, 'dim': 4})},
    'TICA + kmeans': {'features': torsions,
                      'reduce': ('tica', {'lag': 200, 'n': 4}),
                      'cluster': ('kmeans', {'k': 100, 'dim': 4})},
}

# density-based clustering, if the optional package is available
try:
    import hdbscan
    for red in ['pca', 'tica']:
        args = {'n': 4} if red == 'pca' else {'lag': 200, 'n': 4}
        protocols[red.upper() + ' + hdbscan'] = {
            'features': torsions,
            'reduce': (red, args),
            'cluster': ('hdbscan', {'mcs': 500, 'dim': 4})}
except ImportError:
    print("hdbscan not installed: skipping density-based protocols")

Now we run all of them. The seed makes the $k$-means results reproducible. Note that our original `tr` is not modified.

In [ ]:
results = traj.compare_protocols(tr, protocols, seed=0)
print()
for res in results.values():
    print(res)
print("\noriginal untouched:", not hasattr(tr, 'features'), not hasattr(tr, 'distraj'))

### Looking at the density-based clusters
Before building the MSMs, we look at what HDBSCAN does in the two spaces. The left panels show the density of frames in the first two components; the right panels show the same projection coloured by HDBSCAN core (gray: frames that belong to no core, which in the discrete trajectory inherit the previous label). The clustering is done in four dimensions, so cores can overlap in this 2D view.

In the TICA space the metastable regions are well separated and nearly all the frames fall into a core. In the PCA space the density is a single connected distribution without clear gaps, so HDBSCAN can only pick out a couple of dense regions and discards most of the frames.

In [ ]:
if 'PCA + hdbscan' in results and 'TICA + hdbscan' in results:
    import hdbscan
    import matplotlib as mpl

    def core_labels(res):
        """ Raw HDBSCAN labels (-1 = not in any core) for a protocol result,
        recomputed with the parameters stored in its spec. """
        prm = res.spec['cluster'][1]
        dim, mcs = prm['dim'], prm['mcs']
        X = np.vstack([t.features[:, :dim] for t in res.timeseries])
        hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=prm.get('ms') or mcs,
                              prediction_data=True).fit(X)
        labels, _ = hdbscan.approximate_predict(hdb, X)
        return X, labels

    fig, ax = plt.subplots(2, 2, figsize=(8.5, 7.2))
    for row, (name, lab) in enumerate([('PCA + hdbscan', 'PC'), ('TICA + hdbscan', 'TIC')]):
        X, labels = core_labels(results[name])
        ax[row, 0].hist2d(X[:, 0], X[:, 1], bins=[70, 70], cmap='Blues',
                          norm=mpl.colors.LogNorm())
        noncore = labels < 0
        ax[row, 1].scatter(X[noncore, 0], X[noncore, 1], s=2, c='lightgray', rasterized=True)
        sc = ax[row, 1].scatter(X[~noncore, 0], X[~noncore, 1], s=2, c=labels[~noncore],
                                cmap='tab10', vmin=0, vmax=9, rasterized=True)
        for a in ax[row]:
            a.set_xlabel(lab + '1'); a.set_ylabel(lab + '2')
        ncl = len(set(labels[~noncore]))
        ax[row, 0].set_title(name.split()[0] + ': density')
        ax[row, 1].set_title('%s: %i HDBSCAN cores, %.0f%% of frames' %
                             (name.split()[0], ncl, 100*(~noncore).mean()))
    plt.tight_layout()

### Building an MSM for each discretization
For every protocol we build an MSM at a range of lag times. We symmetrize the count matrix (`sym=True`) in all cases, so that the comparison is not affected by differences in reversibility treatment, and we bootstrap the count matrix to obtain error bars on the relaxation times.

In [ ]:
from mastermsm.msm import msm

lags = [10, 20, 50, 100, 200, 500, 1000]
msms = {}
for name, res in results.items():
    t = res.timeseries[0]
    t.find_keys()
    sm = msm.SuperMSM([t], sym=True)
    sm.convergence_test(time=lags, error=True)
    msms[name] = sm

### Implied timescales
The relaxation times of an MSM should be independent of the lag time once it is long enough for the model to be Markovian. A discretization that resolves the slow dynamics well will show flat timescales that are *high*; one that misses part of the slow process will give lower, or still-drifting, values. The shaded area marks the region where the timescale is shorter than the lag time, which cannot be resolved.

In [ ]:
n = len(msms)
fig, ax = plt.subplots(1, n, figsize=(3.4*n, 3.4), sharey=True)
ax = np.atleast_1d(ax)
for a, (name, sm) in zip(ax, msms.items()):
    for i in range(min(4, len(sm.msms[lags[0]].tauT))):
        d = np.array([[x, sm.msms[x].tauT[i], sm.msms[x].tau_std[i]]
                      for x in sorted(sm.msms)])
        a.errorbar(d[:, 0], d[:, 1], yerr=d[:, 2], fmt='o-', ms=5, capsize=2)
    a.fill_between(d[:, 0], 1, d[:, 0], color='lightgray')
    a.set_xscale('log'); a.set_yscale('log')
    a.set_xlabel(r'$\Delta t$ [ps]'); a.set_title(name, fontsize=11)
ax[0].set_ylabel(r'$\tau_i$ [ps]')
ax[0].set_ylim(1e2, 3e4)
plt.tight_layout()

The following table summarizes the outcome at a lag time of 100 ps: the number of states each method produced, how many frames the clustering assigned itself (relevant for density-based methods, where the rest inherit the previous label), and the slowest relaxation times.

In [ ]:
lagt = 100
print("%-16s %7s %9s %17s %17s %17s" %
      ("protocol", "states", "assigned", "tau1 [ps]", "tau2 [ps]", "tau3 [ps]"))
for name, sm in msms.items():
    res = results[name]
    t, s = sm.msms[lagt].tauT, sm.msms[lagt].tau_std
    print("%-16s %7i %9.2f %s" % (name, res.n_states, res.frac_assigned,
          " ".join("%8.0f +- %-5.0f" % (t[i], s[i]) for i in range(min(3, len(t))))))

### How to read the comparison
* The **slowest timescale** is the natural quantity to compare, since a discretization that resolves a slow process should show it as a flat, high plateau. Error bars are bootstrap estimates from resampling the trajectory.
* **BH with A/E/L** and **TICA + $k$-means** agree: both give a flat slowest timescale of about 10 ns (13 $\\pm$ 4 ns and 9.5 $\\pm$ 2 ns at $\\Delta t$ = 100 ps, compatible within the errors), and similar faster modes.
* **BH with A/E only** gives a slowest timescale of about 1 ns, an order of magnitude lower. The reason is that the left-handed region L is not negligible in this trajectory (about 7-11% of the time for residues 3 to 5, and at least one L residue in about 23% of the frames), so leaving it out is not a harmless simplification. Frames in the L region belong to no state when only A and E are defined, and the assignment procedure gives them the state of the previous frame, so the model is partly built on carried-over labels. Note that for this reason the `assigned` column in the table does not show it: it reports 1.00 for both BH variants. We do not identify here which process is the slow one, only that its resolution depends on having the L state.
* **PCA + $k$-means** selects directions by variance, which are not necessarily the slow ones. With the same number of states it gives lower timescales that keep increasing with the lag time, i.e. it is not converged within the range shown.
* **HDBSCAN** (density-based, with `mcs` = 500) behaves very differently depending on the space it works in. In the TICA space it finds 6 core regions covering 98% of the frames and reproduces the same flat slowest timescale of about 9 ns as $k$-means with only 6 states. In the PCA space it finds only 2 states, with just 43% of the frames in cluster cores (the remainder inherit a label), and the model shows a single process of about 0.7 ns. Since the result depends on `mcs`, which we did not optimize, this comparison should be read as indicative: other values give between 8 and 10 states in PCA space and 8 or 9 in TICA space.
* The **error bars** come from bootstrapping and are themselves noisy: they change noticeably between executions of the notebook (the bootstrap is not seeded), particularly for the slowest process at long lag times, where few transitions are available. They are best read as an indication of which differences are clearly significant (for instance A/E against A/E/L, or PCA against TICA) rather than as precise confidence intervals.
* The number of states is not by itself a measure of quality: BH A/E/L has 121 microstates, many sparsely populated, which is reflected in the larger error bars, whereas the $k$-means models have 100 by choice. Comparing the models fairly with a score that penalizes overfitting (cross-validation) is a planned addition.
* **BH** has states with a clear physical meaning, which makes it valuable for interpretation in peptides, but it does not generalize to other systems, for which the feature-based routes are the way to go. TICA reaches the same slow dynamics here without any prior knowledge about the Ramachandran regions.
